# Exercice 0 : échauffement

Q1.	Donnez toutes les séquences d'ARN (AUGC) possibles codant pour la séquence peptidique : TrpGluIleTyr.

---
Toutes les combinaisons possibles entre les éléments suivant en conservant l'ordre:
* Trp : UGG
* Glu : GAA GAG
* Ile : AUU AUC AUA
* Tyr : UAU UAC

# Exercice 1 : biais d’utilisation des codons

Voici en cadeau le code génétique

In [ ]:
genecode = {
    'ATA':'I', 'ATC':'I', 'ATT':'I', 'ATG':'M',
    'ACA':'T', 'ACC':'T', 'ACG':'T', 'ACT':'T',
    'AAC':'N', 'AAT':'N', 'AAA':'K', 'AAG':'K',
    'AGC':'S', 'AGT':'S', 'AGA':'R', 'AGG':'R',
    'CTA':'L', 'CTC':'L', 'CTG':'L', 'CTT':'L',
    'CCA':'P', 'CCC':'P', 'CCG':'P', 'CCT':'P',
    'CAC':'H', 'CAT':'H', 'CAA':'Q', 'CAG':'Q',
    'CGA':'R', 'CGC':'R', 'CGG':'R', 'CGT':'R',
    'GTA':'V', 'GTC':'V', 'GTG':'V', 'GTT':'V',
    'GCA':'A', 'GCC':'A', 'GCG':'A', 'GCT':'A',
    'GAC':'D', 'GAT':'D', 'GAA':'E', 'GAG':'E',
    'GGA':'G', 'GGC':'G', 'GGG':'G', 'GGT':'G',
    'TCA':'S', 'TCC':'S', 'TCG':'S', 'TCT':'S',
    'TTC':'F', 'TTT':'F', 'TTA':'L', 'TTG':'L',
    'TAC':'Y', 'TAT':'Y', 'TAA':'_', 'TAG':'_',
    'TGC':'C', 'TGT':'C', 'TGA':'_', 'TGG':'W',
}


Le code génétique étant redondant, plusieurs codons codent pour un même acide aminé. A l’inverse, cela veut dire que pour stocker un même acide aminé dans le génome, un organisme a différentes possibilités. Dans cet exercice, on va étudier si différents organismes ont des préférences différentes pour coder différents acides aminés ?

Q1. Récupperez les génomes des organismes suivants à partir de [Genbank](https://www.ncbi.nlm.nih.gov/nuccore) :

* AAV2 - NC_001401.2
* Plasmodium Falciparum 3D7 Chromosome 9 – AL844508
* Escherichia coli souche BL21 - NZ_CP053601
* Homo Sapiens Chromosome 10 build GRCh38.p14 - NC_000010.11

---
On va chercher les différents IDs dans la barre de recherche de genbank (lien ci-dessus).
Sur la page de l'entrée correspondant à chaque ID, on va cliquer sur "send to" (en haut a droite) -> **Coding Sequences** -> FASTA Nucleotide -> Create File.

Q2. Calculez la fréquence d’apparition de chaque codon présents dans chacun de ces génomes.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import numpy as np

def read_fasta(fname):
    res = []
    tmp = ""
    with open(fname, "r") as f:
        for line in f:
            if line.startswith(">"):
                if tmp != "":
                    res.append(tmp)
                    tmp = ""
            else:
                tmp += line.rstrip("\n")
    return res

codons = list(genecode.keys())
AAs = "TKQSY_NEPCHVRLWMIFADG"

orgs = ["AL844508_chr9", "NC_000010.11_chr10", "NC_001401.2", "NZ_CP053601"]

revs = {}
for org in orgs:
    seqs = read_fasta("/content/gdrive/MyDrive/" + org + "_coding_sequence.txt")

    rev = np.zeros((len(codons), len(AAs)))

    nseq = 0
    for seq in seqs:
        #I don't want to deal with this, so just discard these sequences
        if len(seq) % 3 != 0:
            continue
        nseq += 1
        for i in range(0,len(seq) - 2,3):
            co = seq[i:(i+3)]
            rev[codons.index(co), AAs.index(genecode[co])] += 1

    nco = int(np.sum(rev))
    rev = rev / rev.sum(axis=0)
    revs[org] = rev

Q3. Affichez les résultats sous forme de heatmap avec en axe x les acides aminés et en axe y les codons. Chaque case représente en niveau de couleur la fréquence d'apparition d'un codon pour un acide aminé.

In [ ]:
from matplotlib import pyplot as plt

for org in orgs:
    plt.figure(figsize=(10,10))
    plt.imshow(revs[org], cmap="Blues")
    plt.colorbar()
    plt.xticks(ticks=range(len(AAs)), labels=AAs)
    plt.yticks(ticks=range(len(codons)), labels=codons)
    plt.title(org + " nseqs={}, ncodons={}".format(nseq, nco))
    plt.tight_layout()
    plt.savefig("codon_usage_bias_" + org + ".png", bbox_inches="tight")

Q4. Ces heatmaps sont-elles similaires ?

---
Ces heatmaps sont assez similaires mais pas identiques. Il existe bien des préférences de codons différentes par espèce. Notez aussi qu'on a un nombre très différent de gènes par espèce (par exemple AAV2 a très peux de gènes), cela peut aussi créer de la variabilité dans les résultats.

Q5. Pour chaque organisme, donnez le codon le plus fréquent codant pour chaque acide aminé.

In [ ]:
for org in orgs:
  print(org)
  for AA in AAs:
    freqs = revs[org][:, AAs.index(AA)]
    print(codons[np.where(freqs == np.max(freqs))[0][0]])

# Exercice 2 : optimisation de codons

L’exercice précédent nous a montré qu’il existait bien un biais dans l’utilisation des codons synonymes entre différents organismes. En pratique, si une séquence d’ADN d’un organisme ne suit pas son biais d'utilisation de codons, cela peut avoir un impact sur sa vitesse de traduction et donc la quantité de protéines associées.

Si on souhaite introduire dans un organisme un gène provenant d’un autre organisme il est donc préférable d’optimiser la séquence selon les biais de codons de l’organisme cible.

Q1. En vous basant sur les tables créées à l'exercice précédent, faites un code qui prend en entrée une séquence d'ADN codante et un nom d'espèce (parmi ceux utilisé à l'exercice précédent) et retourne la séquence produisant la même protéine mais utilisant uniquement les codons les plus fréquemment utilisés par cette espèce.

In [ ]:
seq = "ATGAACATGCCATCGACTCCGCTGGCACCTACCACGGGGACAGCCACCTGCAGCTGGAGCGCATCAACGTGTACTACAACGAGGCCAGCGGTGGCAGGTACGTGCCCCGCGCTGTGCTCGTGGATCTGGAGCCGGGCACCATGGACTCTGTGCGCTCGGGGCCCTTCGGGCAGGTCTTCAGGCCAGACAACTTCATCTTCGGGGAAGCTGCTGTCCTGTAACTCTGGGGGAGGGGGTTTCATCTGCTCCACCTGCAGGGCGAACGGTGCTCTCACCTCACGTGTGACGCTTGGCTCTTTCTGCATTATGGTGGTGACCACTGATGACCGTATACCTGGCCGTCGAGTGACCGGCTGTGCTGTCTTACAGGTGTCGGACACCGTGGTGGAGCCCTACAACGCCACCCTCTCAGTCCACCAGCTCATAGAAAACGCAGATGAGACCTTTTGCATAGATAACGAAGCTCTGTATGACATATGTTCCAAGACCCTAAAACTGCCCACACCCACCTATGGTGACCTGAACCACCTGGTGTCTGCTACCATGAGTGGGGTCACCACGTGCCTGCGCTTCCCGGGCCAGCTGAATGCTGACCTGCGGAAGCTGGCCGTGAACATGGTCCCGTTTCCCCGGCTGCATTTCTTCATGCCCGGCTTTGCCCCACTGACCAGCCGGGGCAGCCAGCAGTACCGGGCCTTGACTGTGGCTGAGCTTACCCAGCAGATGTTTGATGCTAAGAACATGATGGCTGCCTGTGACCCCCGTCACGGCCGCTACCTAACGGCGGCTGCCATTTTCAGGGGTCGCATGCCCATGAGGGAGGTGGATGAACAAATGTTCAACATTCAAGATAAGAACAGCAGTTACTTTGCTGACTGGCTCCCCAACAACGTAAAAACAGCCGTCTGTGACATCCCACCCCGGGGGCTAAAAATGTCAGCCACCTTCATTGGGAATAATACGGCCATCCAGGAACTCTTCAAGCGTGTCTCAGAGCAGTTTACAGCAATGTTCAGGCGCAAGGCCTTCCTCCACTGGTACACGGGCGAGGGCATGGATGAGATGGAATTCACCGAGGCCGAGAGCAACATGAACGACCTGGTGTCTGAATATCAGCAATATCAGGATGCCACGGCCGAGGAGGAGGAGGATGAGGAGTATGCCGAGGAGGAGGTGGCCTAG"
#rev, AAs and codons come from exo 1
org = "AL844508_chr9" #or "NC_000010.11_chr10" "NC_001401.2" "NZ_CP053601"

res = ""
for i in range(0, len(seq)-2, 3):
    co = seq[i:(i+3)]
    AA = genecode[co]
    idx_AA = AAs.index(AA)
    idx_co = np.argmax(revs[org][:,idx_AA])
    res += codons[idx_co]

#make sure sequences have the same size
assert(len(seq) == len(res))
#make sure sequences code for the same AA
for i in range(0,len(seq)-2,3):
    assert(genecode[seq[i:(i+3)]] == genecode[res[i:(i+3)]])

print("Base seq:")
print(seq)
print("---")
print("Optimised for {}:".format(org))
print(res)
print("---")
print("Difference")
print("".join(["." if res[i] == seq[i] else "-" for i in range(len(res))]))